# Training a Neural Network Agent for Tag (REINFORCE)

In this tutorial, we train a neural network policy to control an adversary agent
in the **Tag** environment.

The agent observes:
- its own position,
- relative positions of prey agents,
- relative positions of obstacles,

and receives reward:

\[
r = \frac{1}{\text{distance to closest prey}}
\]

We use the **REINFORCE policy gradient algorithm** to learn a policy
\(\pi_\theta(a \mid s)\) parameterized by a neural network.

Students should not be limited to the REINFORCE algorithm. They could try to implement an actor critic model for example. Go wild

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from pettingzoo.mpe import simple_tag_v3
from pettingzoo_wrapper import AdversaryObsRewardWrapper, make_env
import matplotlib.pyplot as plt
from show_live_tag import plot_learning_curve, render_tag
from tournament_loader import load_default_policies

Below the student groups should change their group name (GROUP_NAME) and the role as predator or prey (AGENT_ROLE)

Also update the total number of prey groups (NUM_PREY) and total number of predator groups (NUM_PREDATORS). This will change the size of the observations so each model will be specific to this setting.

The NUM_EPOCHS should also be changed to however long the training needs to happen

In [ ]:
# ------------------------
# Configuration
# ------------------------
GROUP_NAME = "group_C_predator"   # <<< students change this
AGENT_ROLE = "prey"  # <<< students change this: "predator" or "prey"
NUM_PREY = 1 # number of prey groups
NUM_PREDATORS = 2 # number of predator groups
NUM_EPOCHS = 3000 # <<< students change this: number of training episodes
SAVE_PATH = f"{GROUP_NAME}_{AGENT_ROLE}.pt"
TIMESTEPS_PER_EPISODE = 300 # max timesteps per episode
LEARNING_RATE = 1e-3

Now create an environment based on which tag role the agent takes

In [ ]:
if AGENT_ROLE == "prey":
    agent_id = "agent_0"  # student controls this agent
else:
    agent_id = "adversary_0"  # student controls this agent


# ------------------------
# Create environment
# ------------------------
env = make_env(TIMESTEPS_PER_EPISODE, num_predators=NUM_PREDATORS, num_preys=NUM_PREY)

obs_dim = env.observation_space(agent_id)
act_dim = env.action_space(agent_id)

Create your policy network. This should import PolicyNet from your policy file. This file should be named "group_name"_policy.py. Keep this structure for naming the policy python file as this is how it is called when running the tournament.

The policy variable is just the policy neural network you plan to train. 



In [ ]:
# Load student policy network
from group_A_policy import PolicyNet  # <<< students change this

# ------------------------
# Initialize policy network and optimizer
# ------------------------
policy = PolicyNet(obs_dim, act_dim)
optimizer = optim.Adam(policy.parameters(), lr=LEARNING_RATE)

The block below will load any default agents to train with. If no default agents are found, it will just use a random policy for the other agents. You can rewrite this function later to load your own default trained agents.

In [ ]:
# ------------------------
# Random policy or default policy for other agents
# ------------------------
def random_policy(obs):
    return torch.rand(act_dim) 

default_policies = load_default_policies(env, num_prey=NUM_PREY, num_predators=NUM_PREDATORS, random_policy=random_policy)

Now we introduce the training loop. This loops through NUM_EPOCHS. Each epoch will continue used the done variable is True. This occurs after TIMESTEPS_PER_EPISODE time steps in the environment.

For your agent (agent == agent_id), give the policy an observation, sample an action, and compute a loss. Students should change this loss function for other algorithms.

For all other agents, just sample an action from the policy

Then take a step in the environment with all the agents actions and save rewards from this epoch

The loss is computed from this epoch using the total rewards. loss.backward() computes the gradient and optimizer.step() takes a gradient step for the parameters with the learning rate (LEARNING_RATE). Everything is done behind the scenes by pytorch.

One thing to keep an eye on is that the environment uses numpy but pytorch uses its own torch.tensor type. You can convert between them as we do in torch.tensor(ob, dtype=torch.float32) and sampled.detach().numpy(). The detach just means the variable will be detached from the graph that computes the gradient. So the gradient will not go through this variable. 

In [ ]:
# ------------------------
# Training loop (REINFORCE)
# ------------------------
episode_rewards = []
for episode in range(NUM_EPOCHS):
    obs = env.reset()
    log_probs = []
    rewards = []
    steps = 0

    done = False
    while not done:
        actions = {}

        for agent, ob in obs.items():
            o = torch.tensor(ob, dtype=torch.float32)
            if agent == agent_id:

                action = policy(o)
                dist = torch.distributions.Bernoulli(action)
                sampled = dist.sample()
                log_prob = dist.log_prob(sampled).sum()

                actions[agent] = sampled.detach().numpy()
                log_probs.append(log_prob)
            else:
                actions[agent] = default_policies[agent](o).detach().numpy()

        obs, reward, term, trunc, info = env.step(actions)
        steps += 1
        if not trunc[agent_id]:
            rewards.append(reward[agent_id])
        done = term[agent_id] or trunc[agent_id]

    # Return
    R = sum(rewards)
    episode_rewards.append(R)

    loss = -R * torch.stack(log_probs).sum()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if episode % 50 == 0:
        print(f"Episode {episode}, return = {R:.2f}")

Now save the trained policy and plot your learning curve (your loss over epochs)

In [ ]:
# ------------------------
# Save model
# ------------------------
torch.save(policy.state_dict(), SAVE_PATH)
print("Saved:", SAVE_PATH)

# ------------------------
# Plot training curve
# ------------------------
plot_learning_curve(episode_rewards)

Below, we just show an example run of the learned agent against the default agents.

In [ ]:
# ------------------------
# See example rollouts
# ------------------------
obs = env.reset()
scores = {agent: 0.0 for agent in env.agents}
group_names = {agent: "yours" if agent == agent_id else "CPU" for i, agent in enumerate(env.agents)}
all_policies = {agent: policy if agent == agent_id else default_policies[agent] for agent in env.agents}

render_tag(500, env, scores=scores, group_names=group_names, policies=all_policies)

